In [ ]:
# script to compile calculations from dft/kinetics and dft/thermo

In [1]:
import os
import re
import glob
import yaml
import shutil

import rmgpy.chemkin
import rmgpy.data.kinetics
import rmgpy.data.thermo

import numpy as np
import pandas as pd
# import importlib
# importlib.reload(rmgpy.data.kinetics)

import arkane.ess.gaussian

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
from collections import OrderedDict

import sys
sys.path.append('/projects/westgroup/harris.se/autoscience/reaction_calculator/database/')
import database_fun

DFT_DIR = os.environ['DFT_DIR']
sys.path.append(DFT_DIR)
import autotst_wrapper

import autotst.species
import ase.atoms

sys.path.append('/projects/westgroup/harris.se/autoscience/reaction_calculator/report/')
import report

Loading DFT database from /projects/westgroup/harris.se/autoscience/reaction_calculator/database
hotbit not installed


In [2]:
undecayed_reaction_list = []  # manually add equivalences here [4752]

print(report.N_improve)

10


# Get the changelists

In [3]:
my_chemkin_file = '/projects/westgroup/harris.se/autoscience/fuels/propane/RMG_min/propane_20260326/chem_annotated.inp'

In [4]:
list_include_lists_sp, list_include_lists_rxn = report.get_changelists(my_chemkin_file)

not counting failed (or PDEP) reaction 226


In [5]:
for i in range(len(list_include_lists_sp)):
    print(list_include_lists_rxn[i], list_include_lists_sp[i])

[52, 25650, 31699, 50, 226] [21, 32, 130, 25, 36]
[5691, 23685, 518, 29060, 2232, 203, 202] [14, 49, 48]


In [6]:
total_rxn_include_list = list(set([x for xs in list_include_lists_rxn for x in xs]))
total_sp_include_list = list(set([x for xs in list_include_lists_sp for x in xs]))


# Collect thermo

In [7]:
thermo_libs = glob.glob(os.path.join(DFT_DIR, 'thermo', 'species*', 'arkane', 'RMG_libraries'))
print(f'{len(thermo_libs)} thermo libs')


# Load the Arkane thermo
entries = []
for i, lib_path in enumerate(thermo_libs):
    matches = re.search('species_([0-9]{4})', lib_path)
    species_index = int(matches[1])
    ark_thermo_database = rmgpy.data.thermo.ThermoDatabase()
    ark_thermo_database.load_libraries(
        lib_path,
    )

    for key in ark_thermo_database.libraries['thermo'].entries.keys():
        entry = ark_thermo_database.libraries['thermo'].entries[key]
        entry.index = species_index
        entry.label = entry.item.smiles
        entries.append(entry)
print(f'{len(entries)} entries')

123 thermo libs
123 entries


In [9]:
# compile it all into a single database and a single library which I'll call harris_butane
ark_thermo_database = rmgpy.data.thermo.ThermoDatabase()
ark_thermo_database.libraries['thermo'] = rmgpy.data.thermo.ThermoLibrary()
ark_thermo_database.libraries['thermo'].label = 'harris_propane'
ark_thermo_database.libraries['thermo'].entries = OrderedDict()
found_list = []
for entry in entries:
    # check isomorphism with include_list
    idx = database_fun.get_unique_species_index(rmgpy.species.Species().from_adjacency_list(entry.item.to_adjacency_list()))
    
    
    if idx not in total_sp_include_list:
        continue
        
        
    # discard entry if there wasn't proper separation between modes of vibration and rotation
    logfile = os.path.join(DFT_DIR, 'thermo', f'species_{idx:04}', 'arkane', 'freq.log')
    original_freqs, new_freqs = report.get_projected_freqs(logfile)
    diff_index = len(original_freqs) - len(new_freqs)
    differences = new_freqs - original_freqs[diff_index:]
    error_percents = np.abs(np.divide(differences, original_freqs[diff_index:])) * 100.0
    if np.max(error_percents) > 10.0:
        print(f'Leaving out species {idx} because rotors not properly separated')
        plot_proj_rotors(original_freqs, new_freqs, title=database_fun.index2species(idx).smiles)
        continue
        
    ark_thermo_database.libraries['thermo'].entries[entry.label] = entry
    found_list.append(idx)

In [10]:
print(set(total_sp_include_list) - set(found_list))

set()


In [11]:
set(found_list)

{14, 21, 25, 32, 36, 48, 49, 130}

In [12]:
len(total_sp_include_list)

8

In [13]:
total_sp_include_list

[32, 130, 36, 14, 48, 49, 21, 25]

In [14]:
# save the results
thermo_lib = os.path.join(os.path.dirname(my_chemkin_file), 'thermo')
ark_thermo_database.save_libraries(thermo_lib)

In [15]:
# try reading to test
# Load the new thermo library

thermo_lib = os.path.join(os.path.dirname(my_chemkin_file), 'thermo')
ark_thermo_database = rmgpy.data.thermo.ThermoDatabase()
ark_thermo_database.load_libraries(thermo_lib)
# print(ark_kinetics_database.libraries)
print(f'{len(ark_thermo_database.libraries["harris_propane"].entries)} entries loaded')


8 entries loaded


# Add kinetics

In [16]:
total_rxn_include_list

[226, 29060, 23685, 518, 202, 203, 25650, 31699, 52, 50, 2232, 5691]

In [17]:
len(total_rxn_include_list)

12

In [18]:
# first, get valid kinetics from old workflow
kinetics_libs = glob.glob(os.path.join(DFT_DIR, 'kinetics', 'reaction*', 'arkane', 'RMG_libraries'))

# Load the Arkane kinetics
entries = []
found_rxn_list = []
for i, lib_path in enumerate(kinetics_libs):
    
    matches = re.search('reaction_([0-9]{4,6})', lib_path)
    reaction_index = int(matches[1])
    
    ark_kinetics_database = rmgpy.data.kinetics.KineticsDatabase()
    ark_kinetics_database.load_libraries(lib_path)
    
    
    
    
    # TODO fix bug related to load_libraries not getting the actual name
    for key in ark_kinetics_database.libraries['RMG_libraries'].entries.keys():
        entry = ark_kinetics_database.libraries['RMG_libraries'].entries[key]
        
        
        # check isomorphism with include_list
        idx = database_fun.get_unique_reaction_index(ark_kinetics_database.libraries['RMG_libraries'].entries[key].item)
        if idx not in total_rxn_include_list + undecayed_reaction_list:  # include the un-decayed version of the reaction
            break
        found_rxn_list.append(reaction_index)
        entry.index = reaction_index
        entries.append(entry)
        print(f'Adding\t{entry.index}\t{entry}')

Adding	50	O2(2) + C[C]=O <=> HO2(16) + C2H2O(603)
Adding	52	O2(2) + [CH2]C=O <=> HO2(16) + C2H2O(603)
Adding	203	O2(2) + C[CH2](20) <=> HO2(16) + C2H4(11)
Adding	226	H(14) + C#C <=> [CH]=C
Adding	518	O2(2) + C[O] <=> HO2(16) + CH2O(9)
Adding	23685	H2O2(17) + [CH2]CC(21) <=> HO2(16) + CCC
Adding	25650	OH(15) + CCC <=> H2O(8) + [CH2]CC(21)
Adding	31699	H(14) + CCC <=> H2(13) + [CH2]CC(21)


In [19]:
print(set(total_rxn_include_list) - set(found_rxn_list))

{2232, 202, 5691, 29060}


In [20]:
found_rxn_list

[50, 52, 203, 226, 518, 23685, 25650, 31699]

In [21]:
# compile it all into a single database and a single library which I'll call harris_butane
ark_kinetics_database = rmgpy.data.kinetics.KineticsDatabase()
ark_kinetics_database.libraries['kinetics'] = rmgpy.data.kinetics.KineticsLibrary()
ark_kinetics_database.libraries['kinetics'].label = 'harris_propane'
ark_kinetics_database.libraries['kinetics'].name = 'harris_propane'
ark_kinetics_database.libraries['kinetics'].entries = OrderedDict()
for entry in entries:
    ark_kinetics_database.libraries['kinetics'].entries[entry.label] = entry

In [22]:
# save the results
# output_path = os.path.join(DFT_DIR, 'kinetics')
# ark_kinetics_database.save_libraries(output_path, reindex=False)
kinetics_lib = os.path.join(os.path.dirname(my_chemkin_file), 'harris_kinetics')
ark_kinetics_database.save_libraries(kinetics_lib, reindex=False)

In [23]:
# read the results again
kinetics_lib = os.path.join(os.path.dirname(my_chemkin_file), 'harris_kinetics')
ark_kinetics_database = rmgpy.data.kinetics.KineticsDatabase()
ark_kinetics_database.load_libraries(kinetics_lib)
# print(ark_kinetics_database.libraries)
print(f'{len(ark_kinetics_database.libraries["kinetics"].entries)} entries loaded')

8 entries loaded


# WARNING: THE DESTINATION NAME MUST BE SET MANUALLY

In [ ]:
# transfer the results to the RMG-database

thermo_dest = os.path.join(rmgpy.settings['database.directory'], 'thermo', 'libraries', f'harris_propane_min10_separate_{len(mech_files):04}.py')
print(thermo_dest)

kinetics_dest = os.path.join(rmgpy.settings['database.directory'], 'kinetics', 'libraries', f'harris_propane_min10_separate_{len(mech_files):04}')
print(kinetics_dest)

In [ ]:
print('copying files')
shutil.copyfile(os.path.join(thermo_lib, '/harris_propane.py'), thermo_dest)
shutil.copytree(os.path.join(kinetics_lib, 'kinetics'), kinetics_dest)


In [ ]:
rmgpy.settings['database.directory']